# Results Analysis

Load and visualise results from all four experiment scripts.

In [ ]:
import os, json, pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

sns.set_theme(style='whitegrid')
RESULTS = '../experiments/Results'

## Figure 9 / Table 8 — SubSumE ROUGE + SBERT scores

In [ ]:
def load_subsume_results(exp_folder, model_name, min_r=0, max_r=275, trial=0):
    base = os.path.join(RESULTS, exp_folder, model_name)
    stem = f'{min_r}_{max_r}_trail_{trial}'
    scores = np.load(os.path.join(base, f'all_scores_range_{stem}.npz'))['scores']
    sbert  = np.load(os.path.join(base, f'all_sbert_scores_{stem}.npz'))['scores']
    rts    = np.load(os.path.join(base, f'all_runtimes_{stem}.npz'))['scores']
    return scores, sbert, rts

try:
    scores, sbert, rts = load_subsume_results('Figure9', 'Ex2Bundle')
    avg = np.mean(scores, axis=(0, 1))  # shape: (3, 4) → avg over intents & test docs
    print('Avg ROUGE-1/2/L [p, r, f1, f2]:')
    print(np.round(avg, 4))
    print(f'Avg SBERT: {np.mean(sbert):.4f}')
    print(f'Avg runtime: {np.mean(rts):.4f}s')
except FileNotFoundError as e:
    print(f'Results not found — run run_figure9_subsume.py first: {e}')

## Figure 12 — Quality-function ablation

In [ ]:
variants = ['C_FREQ', 'BS', 'P_FREQ', 'TS', 'ALL']
f1_r1, f1_r2, f1_rL, sbert_all = {}, {}, {}, {}

for v in variants:
    try:
        sc, sb, _ = load_subsume_results('Figure12', f'Ex2Bundle_{v}')
        avg = np.mean(sc, axis=(0, 1))
        f1_r1[v]    = avg[0][2]  # ROUGE-1 F1
        f1_r2[v]    = avg[1][2]  # ROUGE-2 F1
        f1_rL[v]    = avg[2][2]  # ROUGE-L F1
        sbert_all[v] = float(np.mean(sb))
    except FileNotFoundError:
        pass

if f1_r1:
    df = pd.DataFrame({'ROUGE-1': f1_r1, 'ROUGE-2': f1_r2, 'ROUGE-L': f1_rL, 'SBERT': sbert_all}).T
    display(df.round(4))

    fig, ax = plt.subplots(figsize=(8, 4))
    df.loc['ROUGE-1'].plot(kind='bar', ax=ax)
    ax.set_title('ROUGE-1 F1 by Quality-Function Variant (Figure 12)')
    ax.set_ylabel('ROUGE-1 F1')
    plt.tight_layout()
    plt.show()
else:
    print('No Figure 12 results found — run run_figure12_variants.py first')

## Figure 13 — Relaxation frequency

In [ ]:
fig13_pkl = '../results/Figure13/evaluation_data.pkl'
try:
    with open(fig13_pkl, 'rb') as f:
        data13 = pickle.load(f)

    x   = data13['x_values']
    avg = data13['all_avg_relax_freqs_per_sentence_len']

    plt.figure(figsize=(9, 4))
    plt.plot(x[:len(avg)], avg, 'o-', color='orange')
    plt.xlabel('Num sentences in example summary')
    plt.ylabel('Avg relaxation frequency')
    plt.title('Figure 13: Avg Relaxation Frequency vs Example Summary Length')
    plt.xticks(x)
    plt.tight_layout()
    plt.show()
except FileNotFoundError:
    print('Run run_figure13_relaxation.py first')

## Figure 14 — Scalability (runtime vs. example length)

In [ ]:
fig14_pkl = '../results/Figure14/evaluation_data.pkl'
if not os.path.exists(fig14_pkl):
    fig14_pkl = fig13_pkl  # may reuse figure13 pkl

try:
    with open(fig14_pkl, 'rb') as f:
        data14 = pickle.load(f)

    x       = data14['x_values']
    avg_ret = data14['all_avg_retrieval_times_per_sentence_len']
    avg_lrn = data14['all_avg_learning_times_per_sentence_len']

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    if avg_ret:
        axes[0].plot(x[:len(avg_ret)], avg_ret, 'o-', color='red')
        axes[0].set_title('Avg ILP Retrieval Time')
        axes[0].set_xlabel('Example summary length')
        axes[0].set_ylabel('Seconds')
    if avg_lrn:
        axes[1].plot(x[:len(avg_lrn)], avg_lrn, 'o-', color='blue')
        axes[1].set_title('Avg Learning Time')
        axes[1].set_xlabel('Example summary length')
        axes[1].set_ylabel('Seconds')
    plt.suptitle('Figure 14: Scalability')
    plt.tight_layout()
    plt.show()
except FileNotFoundError:
    print('Run run_figure14_scalability.py first')